In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pyodbc as pyodbc

# Exploratory Data Analysis

Understandin the dataset to explore how the data is present in the database and if there is a need of creating some aggregated tables that can help with :
    Vendor Selection ;
    Product Pricing Optimisation

In [2]:
import urllib
from sqlalchemy import create_engine
from sqlalchemy.types import String
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=KHUSHI-GUPTA\\SQLEXPRESS;"
    "DATABASE=Vendor_Management_System;"
    "Trusted_Connection=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)


In [3]:
# -------- checking tables with total number of rows present in my sql db --------
tables=pd.read_sql_query("SELECT name FROM sys.tables", engine)

In [4]:
pd.read_sql_query('Select count(*) from purchases',engine)

,
0,2372474


In [5]:
query = """
SELECT 
    t.name AS table_name,
    SUM(p.rows) AS row_count
FROM sys.tables t
JOIN sys.partitions p
    ON t.object_id = p.object_id
WHERE p.index_id IN (0,1)
GROUP BY t.name
ORDER BY row_count DESC
"""
pd.read_sql_query(query, engine)


,table_name,row_count
0,sales,12825363
1,purchases,2372474
2,end_inventory,224489
3,begin_inventory,206529
4,begin_inventory-checkpoint,206529
5,vendor_sales_summary,32076
6,purchase_prices,12261
7,vendor_invoice,5543
8,query,0
9,sysdiagrams,0


In [6]:
tables = pd.read_sql_query("SELECT name FROM sys.tables", engine)

for table in tables['name']:
    print('-'*50, table, '-'*50)

    query = f"""
    SELECT COUNT(*) AS count
    FROM [{table}]
    """

    df_count = pd.read_sql_query(query, engine)
    print('Count of Records:', df_count['count'].values[0])


-------------------------------------------------- sysdiagrams --------------------------------------------------
Count of Records: 0
-------------------------------------------------- begin_inventory-checkpoint --------------------------------------------------
Count of Records: 206529
-------------------------------------------------- begin_inventory --------------------------------------------------
Count of Records: 206529
-------------------------------------------------- end_inventory --------------------------------------------------
Count of Records: 224489
-------------------------------------------------- purchases --------------------------------------------------
Count of Records: 2372474
-------------------------------------------------- purchase_prices --------------------------------------------------
Count of Records: 12261
-------------------------------------------------- sales --------------------------------------------------
Count of Records: 12825363
-------------

In [7]:
from IPython.display import display

tables = pd.read_sql_query("SELECT name FROM sys.tables", engine)

for table in tables['name']:
    print('-'*50, table, '-'*50)

    query1 = f"""
    SELECT TOP 5 *
    FROM [{table}]
    """
    query = f"""
    SELECT COUNT(*) AS count
    FROM [{table}]
    """

    df_count = pd.read_sql_query(query, engine)
    print('Count of Records:', df_count['count'].values[0])

    top_5 = pd.read_sql_query(query1, engine)
    display(top_5)


-------------------------------------------------- sysdiagrams --------------------------------------------------
Count of Records: 0


,name,principal_id,diagram_id,version,definition


-------------------------------------------------- begin_inventory-checkpoint --------------------------------------------------
Count of Records: 206529


,InventoryId,Store,City,Brand,Description,Size,onHand,Price,startDate
0,33_HORNSEY_1398,33,HORNSEY,1398,Blantons Sgl Barrel Bourbon,750mL,0,53.990000000000002,2024-01-01
1,33_HORNSEY_1415,33,HORNSEY,1415,Old Grand Dad,750mL,10,12.99,2024-01-01
2,33_HORNSEY_1424,33,HORNSEY,1424,Bulleit Bourbon,375mL,0,12.99,2024-01-01
3,33_HORNSEY_1425,33,HORNSEY,1425,Granite Lightning Whiskey,750mL,16,27.989999999999998,2024-01-01
4,33_HORNSEY_1464,33,HORNSEY,1464,Tin Cup Whiskey,750mL,3,24.989999999999998,2024-01-01


-------------------------------------------------- begin_inventory --------------------------------------------------
Count of Records: 206529


...

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.25999999999999,3.4700000000000002,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55000000000001,8.5700000000000003,None
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.59999999999999,4.6100000000000003,None
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.1999999999998,None
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.19999999999999,None


-------------------------------------------------- vendor_sales_summary --------------------------------------------------
Count of Records: 32076


,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchaseRatio
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,36.99,1750,145080,3811251.50,142049,5101919.5,672819.25,260999.20,68601.68,1290668.00,25.30,97.91,133.86
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,28.99,1750,164038,3804041.25,160247,4819073.5,561512.38,294438.66,144929.23,1015032.25,21.06,97.69,126.68
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,24.99,1750,187407,3418303.75,187140,4538120.5,461140.16,343854.09,123780.22,1119816.75,24.68,99.86,132.76
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,22.99,1750,201682,3261198.00,200412,4475973.0,420050.00,368242.81,257032.06,1214775.00,27.14,99.37,137.25
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,29.99,1750,138109,3023206.00,135838,4223107.5,545778.25,249587.81,257032.06,1199901.50,28.41,98.36,139.69


-------------------------------------------------- query  --------------------------------------------------
Count of Records: 0


,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost,GrossProfit,ProfitMargin,StockTurnover,SalestoPurchaseRatio


In [8]:
vendor_sales = pd.read_sql_query('Select * from sales where Vendorno=4466',engine)
vendor_sales

,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-09,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE
1,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-12,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE
2,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-15,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE
3,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-21,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE
4,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-23,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9448,9_BLACKPOOL_5215,9,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-12-21,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE
9449,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-02,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE
9450,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-09,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE
9451,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-23,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE


In [9]:
vendor_info = pd.read_sql_query('Select * from purchase_prices where Vendornumber=4466',engine)
vendor_info

,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,5215,TGI Fridays Long Island Iced,12.99,1750mL,1750,1,9.4100000000000001,4466,AMERICAN VINTAGE BEVERAGE
1,5255,TGI Fridays Ultimte Mudslide,12.99,1750mL,1750,1,9.3499999999999996,4466,AMERICAN VINTAGE BEVERAGE
2,3140,TGI Fridays Orange Dream,14.99,1750mL,1750,1,11.19,4466,AMERICAN VINTAGE BEVERAGE


In [10]:
vendor_invoice = pd.read_sql_query('Select * from vendor_invoice where Vendornumber=4466',engine)
vendor_invoice

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55000000000001,8.5700000000000003,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-19,8207,2023-12-27,2024-02-26,335,3142.3299999999999,16.969999999999999,None
2,4466,AMERICAN VINTAGE BEVERAGE,2024-01-18,8307,2024-01-03,2024-02-18,41,383.35000000000002,1.99,None
3,4466,AMERICAN VINTAGE BEVERAGE,2024-01-27,8469,2024-01-14,2024-03-11,72,673.20000000000005,3.2999999999999998,None
4,4466,AMERICAN VINTAGE BEVERAGE,2024-02-04,8532,2024-01-19,2024-03-15,79,740.21000000000004,3.48,None
5,4466,AMERICAN VINTAGE BEVERAGE,2024-02-09,8604,2024-01-24,2024-03-15,347,3261.3699999999999,17.609999999999999,None
6,4466,AMERICAN VINTAGE BEVERAGE,2024-02-17,8793,2024-02-05,2024-04-02,72,675.36000000000001,3.1699999999999999,None
7,4466,AMERICAN VINTAGE BEVERAGE,2024-03-01,8892,2024-02-12,2024-03-28,117,1096.05,5.1500000000000004,None
8,4466,AMERICAN VINTAGE BEVERAGE,2024-03-07,8995,2024-02-19,2024-04-02,129,1209.27,5.4400000000000004,None
9,4466,AMERICAN VINTAGE BEVERAGE,2024-03-12,9033,2024-02-22,2024-04-16,147,1377.8699999999999,6.6100000000000003,None


In [11]:
purchases = pd.read_sql_query('Select * from purchases where Vendornumber=4466',engine)
purchases

,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.3499999999999996,4,37.399999999999999,1
1,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.4100000000000001,5,47.049999999999997,1
2,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.3499999999999996,6,56.100000000000001,1
3,38_GOULCREST_5215,38,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8207,2023-12-27,2024-01-07,2024-01-19,2024-02-26,9.4100000000000001,6,56.460000000000001,1
4,59_CLAETHORPES_5215,59,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8207,2023-12-27,2024-01-05,2024-01-19,2024-02-26,9.4100000000000001,6,56.460000000000001,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2187,81_PEMBROKE_5215,81,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,13595,2024-12-20,2024-12-29,2025-01-04,2025-02-10,9.4100000000000001,6,56.460000000000001,1
2188,62_KILMARNOCK_5255,62,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,13595,2024-12-20,2024-12-28,2025-01-04,2025-02-10,9.3499999999999996,5,46.75,1
2189,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,13595,2024-12-20,2024-12-28,2025-01-04,2025-02-10,9.4100000000000001,5,47.049999999999997,1
2190,6_GOULCREST_5215,6,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,13595,2024-12-20,2024-12-31,2025-01-04,2025-02-10,9.4100000000000001,6,56.460000000000001,1


In [12]:
purchases= purchases.apply(pd.to_numeric, errors='ignore')


C:\Users\kg808\AppData\Local\Temp\ipykernel_6372\2977387450.py:1: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  purchases= purchases.apply(pd.to_numeric, errors='ignore')


# Purchases Summary Table

In [13]:
purchases.groupby(['Brand','PurchasePrice'])[['Quantity','Dollars']].sum()

,,Quantity,Dollars
Brand,PurchasePrice,,
3140,11.19,4640,51921.60
5215,9.41,4923,46325.43
5255,9.35,6215,58110.25


In [14]:
vendor_invoice = pd.read_sql_query('Select * from vendor_invoice where Vendornumber=4466',engine)
vendor_invoice

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55000000000001,8.5700000000000003,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-19,8207,2023-12-27,2024-02-26,335,3142.3299999999999,16.969999999999999,None
2,4466,AMERICAN VINTAGE BEVERAGE,2024-01-18,8307,2024-01-03,2024-02-18,41,383.35000000000002,1.99,None
3,4466,AMERICAN VINTAGE BEVERAGE,2024-01-27,8469,2024-01-14,2024-03-11,72,673.20000000000005,3.2999999999999998,None
4,4466,AMERICAN VINTAGE BEVERAGE,2024-02-04,8532,2024-01-19,2024-03-15,79,740.21000000000004,3.48,None
5,4466,AMERICAN VINTAGE BEVERAGE,2024-02-09,8604,2024-01-24,2024-03-15,347,3261.3699999999999,17.609999999999999,None
6,4466,AMERICAN VINTAGE BEVERAGE,2024-02-17,8793,2024-02-05,2024-04-02,72,675.36000000000001,3.1699999999999999,None
7,4466,AMERICAN VINTAGE BEVERAGE,2024-03-01,8892,2024-02-12,2024-03-28,117,1096.05,5.1500000000000004,None
8,4466,AMERICAN VINTAGE BEVERAGE,2024-03-07,8995,2024-02-19,2024-04-02,129,1209.27,5.4400000000000004,None
9,4466,AMERICAN VINTAGE BEVERAGE,2024-03-12,9033,2024-02-22,2024-04-16,147,1377.8699999999999,6.6100000000000003,None


In [15]:
vendor_invoice['PONumber'].nunique()

55

In [16]:
vendor_invoice

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55000000000001,8.5700000000000003,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-19,8207,2023-12-27,2024-02-26,335,3142.3299999999999,16.969999999999999,None
2,4466,AMERICAN VINTAGE BEVERAGE,2024-01-18,8307,2024-01-03,2024-02-18,41,383.35000000000002,1.99,None
3,4466,AMERICAN VINTAGE BEVERAGE,2024-01-27,8469,2024-01-14,2024-03-11,72,673.20000000000005,3.2999999999999998,None
4,4466,AMERICAN VINTAGE BEVERAGE,2024-02-04,8532,2024-01-19,2024-03-15,79,740.21000000000004,3.48,None
5,4466,AMERICAN VINTAGE BEVERAGE,2024-02-09,8604,2024-01-24,2024-03-15,347,3261.3699999999999,17.609999999999999,None
6,4466,AMERICAN VINTAGE BEVERAGE,2024-02-17,8793,2024-02-05,2024-04-02,72,675.36000000000001,3.1699999999999999,None
7,4466,AMERICAN VINTAGE BEVERAGE,2024-03-01,8892,2024-02-12,2024-03-28,117,1096.05,5.1500000000000004,None
8,4466,AMERICAN VINTAGE BEVERAGE,2024-03-07,8995,2024-02-19,2024-04-02,129,1209.27,5.4400000000000004,None
9,4466,AMERICAN VINTAGE BEVERAGE,2024-03-12,9033,2024-02-22,2024-04-16,147,1377.8699999999999,6.6100000000000003,None


In [17]:
vendor_invoice.columns

Index(['VendorNumber', 'VendorName', 'InvoiceDate', 'PONumber', 'PODate',
       'PayDate', 'Quantity', 'Dollars', 'Freight', 'Approval'],
      dtype='object')

In [18]:
vendor_sales

,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-09,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE
1,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-12,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE
2,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-15,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE
3,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-21,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE
4,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-23,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9448,9_BLACKPOOL_5215,9,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-12-21,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE
9449,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-02,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE
9450,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-09,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE
9451,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-23,1750.0,1,1.8400000000000001,4466,AMERICAN VINTAGE BEVERAGE


In [19]:
vendor_sales = vendor_sales.apply(pd.to_numeric, errors='ignore')
 

C:\Users\kg808\AppData\Local\Temp\ipykernel_6372\697402466.py:1: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  vendor_sales = vendor_sales.apply(pd.to_numeric, errors='ignore')


# Sales Summary Table

In [20]:
vendor_sales.groupby(['Brand'])[['SalesDollars','SalesPrice','SalesQuantity']].sum()

,SalesDollars,SalesPrice,SalesQuantity
Brand,,,
3140,50531.10,30071.85,3890
5215,60416.49,41542.02,4651
5255,79187.04,51180.60,6096


# Freight Summary Table

In [21]:
purchases= purchases.apply(pd.to_numeric, errors='ignore').round(2)


C:\Users\kg808\AppData\Local\Temp\ipykernel_6372\2799230401.py:1: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  purchases= purchases.apply(pd.to_numeric, errors='ignore').round(2)


In [22]:
# vendor_invoice = pd.read_sql_query('select* from vendor_invoice',engine)
# vendor_invoice= vendor_invoice.apply(pd.to_numeric, errors='ignore').round(2)


In [23]:
freight_summary = pd.read_sql_query('''
with cte as(
select vendornumber ,cast(ROUND(Freight,2)as int) as freight_ from vendor_invoice
)
select vendornumber,sum(freight_) as freight_cost
from cte
group by vendornumber

''',engine)

In [24]:
freight_summary

,vendornumber,freight_cost
0,8004,50267
1,8112,48322
2,1650,192
3,90024,2778
4,6280,12
...,...,...
121,17035,123754
122,3252,61936
123,1265,0
124,1439,0


# Purchaes_prices Summary table

In [25]:
#purchases 
purchase_prices_summary=pd.read_sql_query('''
    select p.vendornumber,
    p.vendorname,
    p.brand,
    pp.volume ,
    pp.price ,
    p.purchaseprice,
    sum(CAST(round(p.quantity,2) AS INT)),
    sum(cast(round(p.dollars,2) as int))as totalpurchasedollars
    from purchases p join purchase_prices pp
    on p.brand=pp.brand
    where cast(round(p.purchaseprice,2)as int)>0
    group by p.vendornumber,p.vendorname,p.brand,
    pp.volume ,
    pp.price ,
    p.purchaseprice
    order by totalpurchasedollars


''',engine)

In [26]:
vendor_sales.columns

Index(['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'SalesQuantity',
       'SalesDollars', 'SalesPrice', 'SalesDate', 'Volume', 'Classification',
       'ExciseTax', 'VendorNo', 'VendorName'],
      dtype='object')

# Sales Summary

In [27]:
sales_summary=pd.read_sql_query(
'''
SELECT 
    vendorno,
    brand,
    CAST(ROUND(SUM(CAST(salesdollars AS FLOAT)), 2) AS INT) AS total_sales_dollar,
    CAST(ROUND(SUM(CAST(salesprice AS FLOAT)), 2) AS INT) AS total_sales_price,
    CAST(ROUND(SUM(CAST(salesquantity AS FLOAT)), 2) AS INT) AS total_sales_quantity,
    CAST(ROUND(SUM(CAST(excisetax AS FLOAT)), 2) AS INT) AS total_excise_tax
FROM sales
GROUP BY vendorno, brand;


''',
    engine

)

In [28]:
vendor_invoice_summmary = pd.read_sql_query('''
        select * from vendor_invoice

''',engine)
vendor_invoice

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55000000000001,8.5700000000000003,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-19,8207,2023-12-27,2024-02-26,335,3142.3299999999999,16.969999999999999,None
2,4466,AMERICAN VINTAGE BEVERAGE,2024-01-18,8307,2024-01-03,2024-02-18,41,383.35000000000002,1.99,None
3,4466,AMERICAN VINTAGE BEVERAGE,2024-01-27,8469,2024-01-14,2024-03-11,72,673.20000000000005,3.2999999999999998,None
4,4466,AMERICAN VINTAGE BEVERAGE,2024-02-04,8532,2024-01-19,2024-03-15,79,740.21000000000004,3.48,None
5,4466,AMERICAN VINTAGE BEVERAGE,2024-02-09,8604,2024-01-24,2024-03-15,347,3261.3699999999999,17.609999999999999,None
6,4466,AMERICAN VINTAGE BEVERAGE,2024-02-17,8793,2024-02-05,2024-04-02,72,675.36000000000001,3.1699999999999999,None
7,4466,AMERICAN VINTAGE BEVERAGE,2024-03-01,8892,2024-02-12,2024-03-28,117,1096.05,5.1500000000000004,None
8,4466,AMERICAN VINTAGE BEVERAGE,2024-03-07,8995,2024-02-19,2024-04-02,129,1209.27,5.4400000000000004,None
9,4466,AMERICAN VINTAGE BEVERAGE,2024-03-12,9033,2024-02-22,2024-04-16,147,1377.8699999999999,6.6100000000000003,None


In [29]:
vendor_sales_summary = pd.read_sql_query("""
WITH FreightSummary AS (
    SELECT
        VendorNumber,
        SUM(TRY_CAST(Freight AS DECIMAL(18,2))) AS FreightCost
    FROM vendor_invoice
    GROUP BY VendorNumber
),

PurchaseSummary AS (
    SELECT
        p.VendorNumber,
        p.VendorName,
        p.Brand,
        p.Description,
        p.PurchasePrice,
        pp.Price AS ActualPrice,
        pp.Volume,
        SUM(TRY_CAST(p.Quantity AS INT)) AS TotalPurchaseQuantity,
        SUM(TRY_CAST(p.Dollars AS DECIMAL(18,2))) AS TotalPurchaseDollars
    FROM purchases p
    JOIN purchase_prices pp
        ON p.Brand = pp.Brand
    WHERE TRY_CAST(p.PurchasePrice AS DECIMAL(18,2)) > 0
    GROUP BY
        p.VendorNumber,
        p.VendorName,
        p.Brand,
        p.Description,
        p.PurchasePrice,
        pp.Price,
        pp.Volume
),

SalesSummary AS (
    SELECT
        VendorNo,
        Brand,
        SUM(TRY_CAST(SalesQuantity AS INT)) AS TotalSalesQuantity,
        SUM(TRY_CAST(SalesDollars AS DECIMAL(18,2))) AS TotalSalesDollars,
        SUM(TRY_CAST(SalesPrice AS DECIMAL(18,2))) AS TotalSalesPrice,
        SUM(TRY_CAST(ExciseTax AS DECIMAL(18,2))) AS TotalExciseTax
    FROM sales
    GROUP BY VendorNo, Brand
)

SELECT
    ps.VendorNumber,
    ps.VendorName,
    ps.Brand,
    ps.Description,
    ps.PurchasePrice,
    ps.ActualPrice,
    ps.Volume,
    ps.TotalPurchaseQuantity,
    ps.TotalPurchaseDollars,
    ss.TotalSalesQuantity,
    ss.TotalSalesDollars,
    ss.TotalSalesPrice,
    ss.TotalExciseTax,
    fs.FreightCost
FROM PurchaseSummary ps
LEFT JOIN SalesSummary ss
    ON ps.VendorNumber = ss.VendorNo
    AND ps.Brand = ss.Brand
LEFT JOIN FreightSummary fs
    ON ps.VendorNumber = fs.VendorNumber
ORDER BY ps.TotalPurchaseDollars DESC


""",engine)


In [30]:
vendor_sales_summary.shape

(10692, 14)

Performance Optimization:

The query involves heavy joins and aggregations on large datasets like sales and purchases.

Storing the pre-aggregated results avoids repeated expensive computations.

Helps in analyzing sales, purchases, and pricing for different vendors and brands.

Future benefits of storing this data for faster Dashboarding & Reporting.

Instead of running expensive queries each time, dashboards can fetch data quickly from vendor_sales_summary.

## DATA CLEANING

In [31]:
vendor_sales_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10692 entries, 0 to 10691
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   VendorNumber           10692 non-null  object 
 1   VendorName             10692 non-null  object 
 2   Brand                  10692 non-null  object 
 3   Description            10692 non-null  object 
 4   PurchasePrice          10692 non-null  object 
 5   ActualPrice            10692 non-null  object 
 6   Volume                 10692 non-null  object 
 7   TotalPurchaseQuantity  10692 non-null  int64  
 8   TotalPurchaseDollars   10692 non-null  float64
 9   TotalSalesQuantity     10514 non-null  float64
 10  TotalSalesDollars      10514 non-null  float64
 11  TotalSalesPrice        10514 non-null  float64
 12  TotalExciseTax         10498 non-null  float64
 13  FreightCost            10691 non-null  float64
dtypes: float64(6), int64(1), object(7)
memory usage: 1.1+ 

In [32]:
vendor_sales_summary.isnull().sum()

VendorNumber               0
VendorName                 0
Brand                      0
Description                0
PurchasePrice              0
ActualPrice                0
Volume                     0
TotalPurchaseQuantity      0
TotalPurchaseDollars       0
TotalSalesQuantity       178
TotalSalesDollars        178
TotalSalesPrice          178
TotalExciseTax           194
FreightCost                1
dtype: int64

In [33]:
vendor_sales_summary = vendor_sales_summary.astype({
    'VendorNumber': 'int32',
    'TotalPurchaseQuantity': 'int32',
    'TotalSalesQuantity': 'Int64',
    'PurchasePrice': 'float32',
    'ActualPrice': 'float32',
    'TotalPurchaseDollars': 'float32',
    'TotalSalesDollars': 'float32',
    'TotalSalesPrice': 'float32',
    'TotalExciseTax': 'float32',
    'FreightCost': 'float32'
})


In [34]:
vendor_sales_summary = vendor_sales_summary.astype({
    'VendorNumber': 'int32',
    'TotalPurchaseQuantity': 'int32',
    'TotalSalesQuantity': 'Int64',
    'VendorName': 'object',
    'Brand': 'int64',
    'Volume': 'object'
})


In [35]:
vendor_sales_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10692 entries, 0 to 10691
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   VendorNumber           10692 non-null  int32  
 1   VendorName             10692 non-null  object 
 2   Brand                  10692 non-null  int64  
 3   Description            10692 non-null  object 
 4   PurchasePrice          10692 non-null  float32
 5   ActualPrice            10692 non-null  float32
 6   Volume                 10692 non-null  object 
 7   TotalPurchaseQuantity  10692 non-null  int32  
 8   TotalPurchaseDollars   10692 non-null  float32
 9   TotalSalesQuantity     10514 non-null  Int64  
 10  TotalSalesDollars      10514 non-null  float32
 11  TotalSalesPrice        10514 non-null  float32
 12  TotalExciseTax         10498 non-null  float32
 13  FreightCost            10691 non-null  float32
dtypes: Int64(1), float32(7), int32(2), int64(1), object(3)

In [36]:
vendor_sales_summary['VendorName'].unique()

array(['BROWN-FORMAN CORP          ', 'MARTIGNETTI COMPANIES',
       'PERNOD RICARD USA          ', 'DIAGEO NORTH AMERICA INC   ',
       'BACARDI USA INC            ', 'JIM BEAM BRANDS COMPANY    ',
       'MAJESTIC FINE WINES        ', 'ULTRA BEVERAGE COMPANY LLP ',
       'STOLI GROUP,(USA) LLC      ', 'PROXIMO SPIRITS INC.       ',
       'MOET HENNESSY USA INC      ', 'CAMPARI AMERICA            ',
       'SAZERAC CO INC             ', 'CONSTELLATION BRANDS INC   ',
       'M S WALKER INC             ', 'SAZERAC NORTH AMERICA INC. ',
       'PALM BAY INTERNATIONAL INC ', 'REMY COINTREAU USA INC     ',
       'SIDNEY FRANK IMPORTING CO  ', 'E & J GALLO WINERY         ',
       'WILLIAM GRANT & SONS INC   ', 'HEAVEN HILL DISTILLERIES   ',
       'DISARONNO INTERNATIONAL LLC', 'EDRINGTON AMERICAS         ',
       'CASTLE BRANDS CORP.        ', 'SOUTHERN WINE & SPIRITS NE ',
       'STE MICHELLE WINE ESTATES  ', 'TRINCHERO FAMILY ESTATES   ',
       'MHW LTD                    ', 'W

In [37]:
vendor_sales_summary['Volume']=vendor_sales_summary['Volume'].astype('float64')

In [38]:
vendor_sales_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10692 entries, 0 to 10691
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   VendorNumber           10692 non-null  int32  
 1   VendorName             10692 non-null  object 
 2   Brand                  10692 non-null  int64  
 3   Description            10692 non-null  object 
 4   PurchasePrice          10692 non-null  float32
 5   ActualPrice            10692 non-null  float32
 6   Volume                 10692 non-null  float64
 7   TotalPurchaseQuantity  10692 non-null  int32  
 8   TotalPurchaseDollars   10692 non-null  float32
 9   TotalSalesQuantity     10514 non-null  Int64  
 10  TotalSalesDollars      10514 non-null  float32
 11  TotalSalesPrice        10514 non-null  float32
 12  TotalExciseTax         10498 non-null  float32
 13  FreightCost            10691 non-null  float32
dtypes: Int64(1), float32(7), float64(1), int32(2), int64(1

In [39]:
vendor_sales_summary.fillna(0,inplace=True)

In [40]:
vendor_sales_summary.head()

,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.270000,36.990002,1750.0,145080,3811251.50,142049,5101919.5,672819.31250,260999.203125,68601.679688
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.190001,28.990000,1750.0,164038,3804041.25,160247,4819073.5,561512.37500,294438.656250,144929.234375
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.240000,24.990000,1750.0,187407,3418303.75,187140,4538120.5,461140.15625,343854.062500,123780.218750
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.170000,22.990000,1750.0,201682,3261198.00,200412,4475973.0,420050.00000,368242.812500,257032.062500
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.889999,29.990000,1750.0,138109,3023206.00,135838,4223107.5,545778.25000,249587.828125,257032.062500


In [41]:
vendor_sales_summary['Description'].unique()

array(['Jack Daniels No 7 Black', "Tito's Handmade Vodka",
       'Absolut 80 Proof', ..., 'Crown Royal Apple',
       'Concannon Glen Ellen Wh Zin', 'The Club Strawbry Margarita'],
      shape=(9651,), dtype=object)

In [42]:
vendor_sales_summary.isnull().sum()

VendorNumber             0
VendorName               0
Brand                    0
Description              0
PurchasePrice            0
ActualPrice              0
Volume                   0
TotalPurchaseQuantity    0
TotalPurchaseDollars     0
TotalSalesQuantity       0
TotalSalesDollars        0
TotalSalesPrice          0
TotalExciseTax           0
FreightCost              0
dtype: int64

In [43]:
vendor_sales_summary['VendorName']=vendor_sales_summary['VendorName'].str.strip()

In [44]:
vendor_sales_summary['VendorName']

0               BROWN-FORMAN CORP
1           MARTIGNETTI COMPANIES
2               PERNOD RICARD USA
3        DIAGEO NORTH AMERICA INC
4        DIAGEO NORTH AMERICA INC
                   ...           
10687              WINE GROUP INC
10688              SAZERAC CO INC
10689    HEAVEN HILL DISTILLERIES
10690    DIAGEO NORTH AMERICA INC
10691        PROXIMO SPIRITS INC.
Name: VendorName, Length: 10692, dtype: object

# Feature Columns

In [45]:
vendor_sales_summary['GrossProfit']=vendor_sales_summary['TotalSalesDollars']-vendor_sales_summary['TotalPurchaseDollars']

In [46]:
vendor_sales_summary['GrossProfit'].min()

np.float32(-52002.78)

In [47]:
vendor_sales_summary['ProfitMargin']=(vendor_sales_summary['GrossProfit']/vendor_sales_summary['TotalSalesDollars'])*100

In [48]:
vendor_sales_summary['StockTurnover']=(vendor_sales_summary['TotalSalesQuantity']/vendor_sales_summary['TotalPurchaseQuantity'])*100

In [49]:
vendor_sales_summary['SalestoPurchaseRatio']=(vendor_sales_summary['TotalSalesDollars']/vendor_sales_summary['TotalPurchaseDollars'])*100

In [50]:
vendor_sales_summary.columns

Index(['VendorNumber', 'VendorName', 'Brand', 'Description', 'PurchasePrice',
       'ActualPrice', 'Volume', 'TotalPurchaseQuantity',
       'TotalPurchaseDollars', 'TotalSalesQuantity', 'TotalSalesDollars',
       'TotalSalesPrice', 'TotalExciseTax', 'FreightCost', 'GrossProfit',
       'ProfitMargin', 'StockTurnover', 'SalestoPurchaseRatio'],
      dtype='object')

In [51]:
from sqlalchemy import text

create_table_query = text("""
IF NOT EXISTS (
    SELECT * FROM sys.tables WHERE name = 'vendor_sales_summary'
)
BEGIN
    CREATE TABLE vendor_sales_summary (
        VendorNumber INT,
        VendorName VARCHAR(100),
        Brand INT,
        Description VARCHAR(100),
        PurchasePrice DECIMAL(10,2),
        ActualPrice DECIMAL(10,2),
        Volume INT,
        TotalPurchaseQuantity INT,
        TotalPurchaseDollars DECIMAL(15,2),
        TotalSalesQuantity INT,
        TotalSalesDollars DECIMAL(15,2),
        TotalSalesPrice DECIMAL(15,2),
        TotalExciseTax DECIMAL(15,2),
        FreightCost DECIMAL(15,2),
        GrossProfit DECIMAL(15,2),
        ProfitMargin DECIMAL(15,2),
        StockTurnover DECIMAL(15,2),
        SalesToPurchaseRatio DECIMAL(15,2),
        PRIMARY KEY (VendorNumber, Brand)
    )
END
""")

with engine.begin() as conn:
    conn.execute(create_table_query)


In [52]:
pd.read_sql_query('''select * from vendor_sales_summary''',engine)

,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchaseRatio
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,36.99,1750,145080,3811251.50,142049,5101919.50,672819.25,260999.20,68601.68,1290668.00,25.30,97.91,133.86
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,28.99,1750,164038,3804041.25,160247,4819073.50,561512.38,294438.66,144929.23,1015032.25,21.06,97.69,126.68
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,24.99,1750,187407,3418303.75,187140,4538120.50,461140.16,343854.09,123780.22,1119816.75,24.68,99.86,132.76
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,22.99,1750,201682,3261198.00,200412,4475973.00,420050.00,368242.81,257032.06,1214775.00,27.14,99.37,137.25
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,29.99,1750,138109,3023206.00,135838,4223107.50,545778.25,249587.81,257032.06,1199901.50,28.41,98.36,139.69
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32071,9815,WINE GROUP INC,8527,Concannon Glen Ellen Wh Zin,1.32,4.99,750,2,2.64,5,15.95,10.96,0.55,27100.41,13.31,83.45,250.00,604.17
32072,8004,SAZERAC CO INC,5683,Dr McGillicuddy's Apple Pie,0.39,0.49,50,6,2.34,134,65.66,1.47,7.04,50293.62,63.32,96.44,1000.00,1000.00
32073,3924,HEAVEN HILL DISTILLERIES,9123,Deep Eddy Vodka,0.74,0.99,50,2,1.48,2,1.98,0.99,0.10,14069.87,0.50,25.25,100.00,133.78
32074,3960,DIAGEO NORTH AMERICA INC,6127,The Club Strawbry Margarita,1.47,1.99,200,1,1.47,72,143.28,77.61,15.12,257032.06,141.81,98.97,1000.00,1000.00


In [53]:
import numpy as np

# Replace inf/-inf with NULL-safe values
vendor_sales_summary.replace([np.inf, -np.inf], np.nan, inplace=True)

# Fill NaNs AFTER replacement
vendor_sales_summary.fillna(0, inplace=True)

# Cap extreme KPI values (business-safe limits)
vendor_sales_summary['StockTurnover'] = vendor_sales_summary['StockTurnover'].clip(0, 1000)
vendor_sales_summary['SalestoPurchaseRatio'] = vendor_sales_summary['SalestoPurchaseRatio'].clip(0, 1000)
vendor_sales_summary['ProfitMargin'] = vendor_sales_summary['ProfitMargin'].clip(-100, 100)

# Round all floats
float_cols = vendor_sales_summary.select_dtypes(include=['float']).columns
vendor_sales_summary[float_cols] = vendor_sales_summary[float_cols].round(2)


In [54]:
engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)


In [55]:
vendor_sales_summary.to_sql(
    'vendor_sales_summary',
    engine,
    schema='dbo',
    if_exists='append',
    index=False,
    chunksize=500,
    method=None      # 🔥 THIS FIXES YOUR ERROR
)


-22

In [56]:
vendor_sales_summary.to_csv(
    'vendor_sales_summary.csv', 
    index=False,      # Prevents writing row numbers (indices) to the file
    encoding='utf-8'  # Ensures special characters are saved correctly
)